In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    print(BASE_DIR)


# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Ambiente Locale rilevato. Procedo con l'esecuzione...
d:\GitHub repositories\crop-spatial-classification
✅ Collegamento ai dati riuscito! Cartella raw: d:\GitHub repositories\crop-spatial-classification\data\raw


In [ ]:
from pathlib import Path

# lista espandibile di anni target per l'estrazione dei dati
TARGET_YEARS = ["2023"]

main_directory = Path(f"{DATA_DIR}/raw/crops_types_yearly_capitanata_03035")
tifs_3035 = {}

for year in TARGET_YEARS:
    year_dir = main_directory / year
    if year_dir.is_dir():
        # trova tutti i .tif per questo anno
        tifs_3035[year] = [str(tif) for tif in year_dir.rglob("*.tif")]
        print(f"Anno {year}: trovati {len(tifs_3035[year])} file .tif")
    else:
        print(f"Cartella anno {year} non trovata in {main_directory}")

In [ ]:
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

tifs_4326 = {}

for year, file_paths in tifs_3035.items():
    year_file_list = []

    for row_path in file_paths:
        # costruisce il percorso del file riproiettato in EPSG:4326
        file_name = row_path.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # apre il file GeoTIFF originale (EPSG:3035)
        raster_3035 = rioxarray.open_rasterio(row_path)

        # riproiezione in EPSG:4326, usando NEAREST per preservare i codici delle colture
        raster_4326 = raster_3035.rio.reproject("EPSG:4326", resampling=Resampling.nearest)

        # salva il file riproiettato
        raster_4326.rio.to_raster(file_name)
        raster_3035.close()

        year_file_list.append(file_name)
        
    tifs_4326[year] = year_file_list

print(f"\nRiproiezione completata per {sum(len(v) for v in tifs_4326.values())} file")

In [ ]:
from pathlib import Path
import rasterio
import numpy as np

# num. max di punti casuali da estrarre da ciascun file .tif
MAX_SAMPLES_PER_TIF = 5

points = []

# itera su tutti gli anni target e sui rispettivi .tif riproiettati
for year, file_list in tifs_4326.items():
    print(f"Campionamento in corso per l'anno {year} ({len(file_list)} file)")

    for tif_path in file_list:
        with rasterio.open(tif_path) as dataset:
            # legge la matrice 2D dei pixel (banda 1)
            crop_matrix = dataset.read(1)

            # crea una maschera per scartare i pixel nulli o nodata
            # (0 = fuori confine, >= 65534 = nodata)
            valid_mask = (crop_matrix > 0) & (crop_matrix < 65534)

            # trova gli indici dei pixel validi
            rows, cols = np.where(valid_mask)

            if len(rows) == 0:
                continue

            # estrae a caso un campione di pixel senza ripetizioni
            n_samples = min(MAX_SAMPLES_PER_TIF, len(rows))
            indices = np.random.choice(len(rows), size=n_samples, replace=False)

            # converte i pixel selezionati in coordinate GPS geografiche
            for index in indices:
                row, col = rows[index], cols[index]
                lon, lat = dataset.xy(row, col)

                points.append({
                    "year": year.item(),                    # anno coltura
                    "lon": lon.item(),                      # longitudine
                    "lat": lat.item(),                      # latitudine
                    "code": crop_matrix[row, col].item()    # codice coltura
                })

print(f"\nEstrazione completata! Totale punti estratti: {len(points)}")

In [ ]:
import json
from pathlib import Path

# Percorso del file JSON nella cartella di lavoro
json_path = Path(f'{DATA_DIR}/interim/points.json')

# Salva i punti (formato: [["nome_file", lon, lat], ...])
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(points, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(points)} punti in {json_path.resolve()}")

In [3]:
import pandas as pd
df = pd.read_json(f'{DATA_DIR}/interim/points.json')
conteggio = df['code'].value_counts()
print(conteggio)

code
2200    148
1110    105
1120     42
2100     34
1210     20
2310     20
1220     19
1130      3
1150      3
3100      2
1410      2
2320      1
1310      1
Name: count, dtype: int64


In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

xml_path = Path(f"{DATA_DIR}/raw/crops_types_yearly_capitanata_03035/2017/CLMS_HRLVLCC_CTY_S2017_R10m_E47N20_03035_V01_R00/CLMS_HRLVLCC_CTY_S2017_R10m_E47N20_03035_V01_R00.tif.aux.xml")

tree = ET.parse(xml_path)
root = tree.getroot()

legend = {}

for row in root.findall(".//Row"):
    colonne_f = row.findall("F")
    
    crop_id = int(colonne_f[0].text)     
    crop_name = colonne_f[2].text
    
    legend[crop_id] = crop_name

In [ ]:
import json
from pathlib import Path

# Percorso del file JSON nella cartella di lavoro
json_path = Path(f'{DATA_DIR}/processed/legend.json')

# Salva i punti (formato: [["nome_file", lon, lat], ...])
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(legend, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(legend)} punti in {json_path.resolve()}")